# Data Preparation

### Imports and constants

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# shared data-acquisition functions live in scripts/download.py
# (paths there are computed from the module location, so cwd doesn't matter)
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
from download import (
    RAW_DIR, PROCESSED_DIR, CDR_PATH, EXPR_LOG2_THRESHOLD,
    download_cdr, download_xena_matrix, gene_annotation,
)

COHORTS = ["LUSC"]

### Download utilities

RNA-seq + TCGA-CDR acquisition lives in `scripts/download.py` (imported above), reused by `scripts/run_cohort.py` so there is a single source of truth.

We pull RNA-seq from the **UCSC Xena GDC hub**, which mirrors the GDC harmonized STAR-Counts as one pre-assembled matrix per project (`TCGA-{COHORT}.star_fpkm-uq.tsv.gz`) — one ~150 MB download instead of ~500 per-file `gdc-client` transfers. Values are `log2(fpkm_uq_unstranded + 1)`, verified **bit-identical** (max|diff| 1.8e-15, corr 1.0 across all 60,660 genes) to the GDC `fpkm_uq_unstranded` column the pipeline used previously.

# Data download and initial filters

The Xena matrix is genes (rows, versioned Ensembl IDs) × samples (columns, TCGA barcodes). Everything we need is in the barcode plus a committed gene annotation.

### Step 1. Barcode → patient / TSS / sample-type

Each column is a sample barcode `TCGA-XX-XXXX-01A`:

| field | meaning |
|---|---|
| `TCGA-XX-XXXX` | patient barcode — join key to TCGA-CDR |
| field 2 (`XX`) | `tss` — tissue source site (the institution) |
| field 4 (`01`) | sample-type code: `01` = Primary Solid Tumor |

### Step 2. Protein-coding + primary-tumour filter

Xena carries no `gene_type`, so we filter to protein-coding genes via `resources/gene_annotation.tsv` (gencode v36, committed; identical across cohorts). We keep sample-type `01` (primary tumour) — matching the old GDC "Primary Tumor" manifest filter (note this excludes LAML `03` / SKCM `06`, as before).

### Step 3. Deduplicate to one sample per patient

Some patients have >1 primary-tumour sample (vials/repeat extractions). Survival labels are patient-level, so keep the alphabetically-first sample barcode per patient.

### Step 4. Join TCGA-CDR survival labels

Source: `data/raw/tcga_cdr.xlsx`, sheet `TCGA-CDR` (Liu et al. 2018). Join on `bcr_patient_barcode == patient_barcode`.

In [ ]:
# Download the common TCGA-CDR file, then the Xena matrix per cohort
download_cdr()

annot = gene_annotation()
protein_coding = set(annot.loc[annot["gene_type"] == "protein_coding", "gene_id"])

# barcode helpers: TCGA-XX-XXXX-01A
patient = lambda bc: "-".join(bc.split("-")[:3])
tss     = lambda bc: bc.split("-")[1]
stype   = lambda bc: bc.split("-")[3][:2] if len(bc.split("-")) >= 4 else ""

for cohort in COHORTS:
    print(f"Downloading Xena matrix for cohort: {cohort}")
    gz = download_xena_matrix(cohort)

    # genes x samples, values = log2(fpkm_uq + 1)
    mat = pd.read_csv(gz, sep="\t", index_col=0)
    mat = mat[mat.index.isin(protein_coding)]              # protein-coding genes only

    # primary-tumour samples (sample-type 01), then dedup to one sample per patient
    prim = [c for c in mat.columns if stype(c) == "01"]
    mat = mat[prim]
    seen, keep = set(), []
    for c in sorted(mat.columns):
        if patient(c) not in seen:
            seen.add(patient(c)); keep.append(c)
    gene_matrix = mat[keep]                                 # genes x patients (log2 values)

    # per-sample meta, keyed by sample barcode
    dedup = pd.DataFrame({
        "sample_barcode": keep,
        "patient_barcode": [patient(c) for c in keep],
        "tss": [tss(c) for c in keep],
    })
    print(f"\n{len(prim)} primary-tumour samples -> {len(keep)} patients (deduped)")

    # survival labels
    tcga_cdr = pd.read_excel(CDR_PATH, sheet_name="TCGA-CDR")
    cohort_cdr = tcga_cdr[tcga_cdr["type"] == cohort].copy()
    n_matched = dedup["patient_barcode"].isin(cohort_cdr["bcr_patient_barcode"]).sum()
    print(f"patients with survival labels: {n_matched} / {len(dedup)} "
          f"(dropped {len(dedup) - n_matched} with missing/no survival data)")

# Data Preprocessing

## Gene filter, standardise

Xena values are already `log2(fpkm_uq + 1)`, so there is **no log step** here.

- Keep genes with `log2(fpkm_uq+1) >= 1` (equivalently `fpkm_uq >= 1`) in at least 10% of patients.
- Standardise each gene to mean 0, std 1 across patients **within this cohort**.

In [ ]:
# expression filter on the log2(fpkm_uq+1) values:
#   log2(x+1) >= 1  <=>  fpkm_uq >= 1   (same threshold as the old raw-FPKM-UQ filter)
# also removes all-zero / zero-variance genes that would NaN on standardisation
n = gene_matrix.shape[1]
keep = (gene_matrix >= EXPR_LOG2_THRESHOLD).sum(axis=1) >= 0.1 * n
logm = gene_matrix.loc[keep]
print("genes after expression filter:", logm.shape[0])

# standardise each gene across patients (mean 0, std 1) — Xena is already log2
zscore = logm.sub(logm.mean(axis=1), axis=0).div(logm.std(axis=1), axis=0)
assert not zscore.isna().any().any(), "NaNs in zscore — a zero-variance gene slipped through"

In [ ]:

# 1. transpose -> samples x genes
X = zscore.T
X.index.name = "sample_barcode"

# 2. attach patient_barcode + tss from dedup (keyed by sample barcode)
meta = dedup.set_index("sample_barcode")[["patient_barcode", "tss"]]
X = meta.join(X)                       # rows still keyed by sample_barcode

# 3. join survival labels from TCGA-CDR (all endpoints; NaNs kept by design)
labels = (
    cohort_cdr[["bcr_patient_barcode", "OS", "OS.time", "DSS", "DSS.time", "PFI", "PFI.time", "DFI", "DFI.time"]]
    .rename(columns={
        "bcr_patient_barcode": "patient_barcode",
        "OS.time": "OS_time",
        "DSS.time": "DSS_time",
        "PFI.time": "PFI_time",
        "DFI.time": "DFI_time"
    })
)
final = X.merge(labels, on="patient_barcode", how="inner")
print(f"patients: {len(X)} gene-matrix -> {len(final)} after survival join "
      f"(dropped {len(X) - len(final)})")

# 4. order columns: ids, then endpoint labels, then genes
id_cols    = ["patient_barcode", "tss"]
label_cols = ["OS", "OS_time", "DSS", "DSS_time", "PFI", "PFI_time", "DFI", "DFI_time"]
gene_cols  = [c for c in final.columns if c.startswith("ENSG")]
final = final[id_cols + label_cols + gene_cols]

# 5. sanity checks — ids + genes must be complete; endpoint labels may be missing by design
assert final["patient_barcode"].is_unique, "duplicate patients in final table"
assert not final[id_cols + gene_cols].isna().any().any(), "NaNs in ids or gene matrix"
print("missing per endpoint:\n", final[label_cols].isna().sum())

# 6. write -> per-cohort folder data/processed/{cohort}/{cohort}.parquet
cohort_out = PROCESSED_DIR / cohort
cohort_out.mkdir(parents=True, exist_ok=True)
final.to_parquet(cohort_out / f"{cohort}.parquet", index=False)
print("final table:", final.shape, "->", cohort_out / f"{cohort}.parquet")